In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D13 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D13 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import re
import unicodedata

import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D13"
DOCUMENT_NAME = (
    "Eurostat — Unemployment rates by country of birth"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_SHA256 = (
    "ce230f6b59a15af159b50d831a7adad9b1063c57b3f6e6ef86f1979f8484ab89"
)

EXPECTED_RECORD_COUNT = 75

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_YEAR_COUNTS = {
    2020: 25,
    2022: 25,
    2024: 25
}

EXPECTED_CATEGORY = (
    "Labour market time series"
)

EXPECTED_TOPIC = (
    "Unemployment rate by country of birth"
)

EXPECTED_UNIT = "percent"

EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY:
        EXPECTED_RECORD_COUNT
}

EXPECTED_FLAG_COUNTS = {
    "d": 10,
    "b": 5,
    "u": 2
}

EXPECTED_FLAGGED_RECORDS = 17

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

# FROZEN FROM D13 VALIDATION A.
ALIGNMENT_IDENTITY_FIELDS = [
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period"
]

OUTPUT_DIR = Path(
    "outputs_D13_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PATHS = {
    "detailed":
        OUTPUT_DIR
        / "D13_branch_B_validation_detailed.csv",

    "fully_correct":
        OUTPUT_DIR
        / "D13_branch_B_fully_correct_records.csv",

    "discrepant":
        OUTPUT_DIR
        / "D13_branch_B_discrepant_records.csv",

    "missing":
        OUTPUT_DIR
        / "D13_branch_B_missing_records.csv",

    "unsupported":
        OUTPUT_DIR
        / "D13_branch_B_unsupported_records.csv",

    "field_validation":
        OUTPUT_DIR
        / "D13_branch_B_field_validation.csv",

    "category_metrics":
        OUTPUT_DIR
        / "D13_branch_B_category_metrics.csv",

    "year_metrics":
        OUTPUT_DIR
        / "D13_branch_B_year_metrics.csv",

    "alignment_issues":
        OUTPUT_DIR
        / "D13_branch_B_alignment_issues.json",

    "reference_semantics":
        OUTPUT_DIR
        / "D13_reference_semantics_confirmation.json",

    "summary":
        OUTPUT_DIR
        / "D13_branch_B_validation_summary.json"
}

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)
print(
    "Alignment identity fields:",
    ALIGNMENT_IDENTITY_FIELDS
)
print(
    "Primary correctness fields:",
    PRIMARY_CORRECTNESS_FIELDS
)

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D13"
DOCUMENT_NAME = (
    "Eurostat — Unemployment rates by country of birth"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_SHA256 = (
    "ce230f6b59a15af159b50d831a7adad9b1063c57b3f6e6ef86f1979f8484ab89"
)

EXPECTED_RECORD_COUNT = 75

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_YEAR_COUNTS = {
    2020: 25,
    2022: 25,
    2024: 25
}

EXPECTED_CATEGORY = (
    "Labour market time series"
)

EXPECTED_TOPIC = (
    "Unemployment rate by country of birth"
)

EXPECTED_UNIT = "percent"

EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY:
        EXPECTED_RECORD_COUNT
}

EXPECTED_FLAG_COUNTS = {
    "d": 10,
    "b": 5,
    "u": 2
}

EXPECTED_FLAGGED_RECORDS = 17

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

# FROZEN FROM D13 VALIDATION A.
ALIGNMENT_IDENTITY_FIELDS = [
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period"
]

OUTPUT_DIR = Path(
    "outputs_D13_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PATHS = {
    "detailed":
        OUTPUT_DIR
        / "D13_branch_B_validation_detailed.csv",

    "fully_correct":
        OUTPUT_DIR
        / "D13_branch_B_fully_correct_records.csv",

    "discrepant":
        OUTPUT_DIR
        / "D13_branch_B_discrepant_records.csv",

    "missing":
        OUTPUT_DIR
        / "D13_branch_B_missing_records.csv",

    "unsupported":
        OUTPUT_DIR
        / "D13_branch_B_unsupported_records.csv",

    "field_validation":
        OUTPUT_DIR
        / "D13_branch_B_field_validation.csv",

    "category_metrics":
        OUTPUT_DIR
        / "D13_branch_B_category_metrics.csv",

    "year_metrics":
        OUTPUT_DIR
        / "D13_branch_B_year_metrics.csv",

    "alignment_issues":
        OUTPUT_DIR
        / "D13_branch_B_alignment_issues.json",

    "reference_semantics":
        OUTPUT_DIR
        / "D13_reference_semantics_confirmation.json",

    "summary":
        OUTPUT_DIR
        / "D13_branch_B_validation_summary.json"
}

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)
print(
    "Alignment identity fields:",
    ALIGNMENT_IDENTITY_FIELDS
)
print(
    "Primary correctness fields:",
    PRIMARY_CORRECTNESS_FIELDS
)

In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b""
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(
    REFERENCE_FILE
)


reference_df = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig",
    keep_default_na=False
)


with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:

    extraction_payload = json.load(f)


with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:

    technical_diagnostics = (
        json.load(f)
    )


for artefact_name, artefact in {
    "parsed extraction":
        extraction_payload,

    "technical diagnostics":
        technical_diagnostics
}.items():

    if (
        artefact.get(
            "document_id"
        )
        != DOCUMENT_ID
    ):
        raise ValueError(
            f"Unexpected {artefact_name} document_id."
        )

    if (
        artefact.get(
            "branch"
        )
        != BRANCH
    ):
        raise ValueError(
            f"Unexpected {artefact_name} branch."
        )


extracted_records = (
    extraction_payload[
        "records"
    ]
)

extracted_df = pd.DataFrame(
    extracted_records
)


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        REFERENCE_SHA256,

    "reference_sha_matches_stage_1":
        (
            REFERENCE_SHA256
            == EXPECTED_REFERENCE_SHA256
        ),

    "parsed_extraction_file":
        PARSED_EXTRACTION_FILE,

    "parsed_extraction_sha256":
        sha256_file(
            PARSED_EXTRACTION_FILE
        ),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,

    "technical_diagnostics_sha256":
        sha256_file(
            TECHNICAL_DIAGNOSTICS_FILE
        )
}


print(
    "Reference records:",
    len(reference_df)
)

print(
    "Extracted records:",
    len(extracted_records)
)

In [ ]:
# ============================================================
# 4. Confirm fixed Stage 1 D13 reference semantics
# ============================================================

def normalise_text(value):

    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def canonical_period(value):

    text = normalise_text(
        value
    )

    if text is None:
        return None

    if re.fullmatch(
        r"\d{4}",
        text
    ):
        return int(text)

    return text


def canonical_source_location(
    value
):

    text = normalise_text(
        value
    )

    if text is None:
        return None

    match = re.fullmatch(
        (
            r"(?i)sheet\s+(\d+)\s*,\s*"
            r"cell\s+([A-Z]+)(\d+)"
        ),
        text
    )

    if not match:
        return text

    return (
        f"Sheet {int(match.group(1))}, "
        f"cell "
        f"{match.group(2).upper()}"
        f"{int(match.group(3))}"
    )


reference_schema_exact = (
    reference_df.columns.tolist()
    == EXPECTED_FIELDS
)

reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

reference_category_constant = bool(
    (
        reference_df[
            "Category"
        ]
        == EXPECTED_CATEGORY
    ).all()
)

reference_topic_constant = bool(
    (
        reference_df[
            "Topic"
        ]
        == EXPECTED_TOPIC
    ).all()
)

reference_unit_constant = bool(
    (
        reference_df[
            "Unit"
        ]
        == EXPECTED_UNIT
    ).all()
)


reference_values_numeric_series = (
    pd.to_numeric(
        reference_df[
            "Value"
        ],
        errors="coerce"
    )
)

reference_values_numeric = bool(
    reference_values_numeric_series
    .notna()
    .all()
)

reference_values_in_percent_range = bool(
    (
        (
            reference_values_numeric_series
            >= 0
        )
        &
        (
            reference_values_numeric_series
            <= 100
        )
    ).all()
)


reference_periods = (
    reference_df[
        "Reporting Period"
    ]
    .apply(
        canonical_period
    )
)

reference_year_counts = dict(
    Counter(
        reference_periods
    )
)

reference_year_counts_valid = (
    reference_year_counts
    == EXPECTED_YEAR_COUNTS
)


reference_source_locations = (
    reference_df[
        "Source Location"
    ]
    .apply(
        canonical_source_location
    )
)

reference_source_location_format_valid = bool(
    reference_source_locations
    .astype(str)
    .str.fullmatch(
        r"Sheet [1-5], cell [A-Z]+\d+"
    )
    .all()
)

reference_identity_unique = (
    reference_source_locations
    .nunique()
    == EXPECTED_RECORD_COUNT
)


description_required_labels = [
    "Geography:",
    "Sex:",
    "Age class:",
    "Country/region of birth:"
]

reference_description_structure_valid = bool(
    reference_df[
        "Description"
    ].apply(
        lambda text:
            all(
                label in str(text)
                for label
                in description_required_labels
            )
    ).all()
)


flag_pattern = re.compile(
    (
        r";\s*Statistical flag:"
        r"\s*([^;]+)\s*$"
    )
)

observed_reference_flags = []

for description in (
    reference_df[
        "Description"
    ].astype(str)
):

    match = flag_pattern.search(
        description
    )

    if match:

        observed_reference_flags.append(
            match.group(1).strip()
        )


reference_flag_counts = dict(
    Counter(
        observed_reference_flags
    )
)

reference_flag_count_valid = (
    len(
        observed_reference_flags
    )
    == EXPECTED_FLAGGED_RECORDS
)

reference_flag_values_valid = (
    reference_flag_counts
    == EXPECTED_FLAG_COUNTS
)

reference_sha_matches_stage_1 = (
    REFERENCE_SHA256
    == EXPECTED_REFERENCE_SHA256
)


reference_semantic_checks = {
    "reference_schema_exact":
        bool(
            reference_schema_exact
        ),

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_category_counts_valid":
        bool(
            reference_category_counts_valid
        ),

    "reference_category_constant":
        bool(
            reference_category_constant
        ),

    "reference_topic_constant":
        bool(
            reference_topic_constant
        ),

    "reference_unit_constant":
        bool(
            reference_unit_constant
        ),

    "reference_values_numeric":
        bool(
            reference_values_numeric
        ),

    "reference_values_in_percent_range":
        bool(
            reference_values_in_percent_range
        ),

    "reference_year_counts_valid":
        bool(
            reference_year_counts_valid
        ),

    "reference_source_location_format_valid":
        bool(
            reference_source_location_format_valid
        ),

    "reference_identity_unique":
        bool(
            reference_identity_unique
        ),

    "reference_description_structure_valid":
        bool(
            reference_description_structure_valid
        ),

    "reference_flag_count_valid":
        bool(
            reference_flag_count_valid
        ),

    "reference_flag_values_valid":
        bool(
            reference_flag_values_valid
        ),

    "reference_sha_matches_stage_1":
        bool(
            reference_sha_matches_stage_1
        )
}


reference_semantics_valid = all(
    reference_semantic_checks
    .values()
)


REFERENCE_SEMANTICS_CONFIRMATION = {
    "document_id":
        DOCUMENT_ID,

    "reference_semantics_valid":
        bool(
            reference_semantics_valid
        ),

    "checks":
        reference_semantic_checks,

    "observed_flag_counts":
        reference_flag_counts,

    "observed_year_counts":
        reference_year_counts
}


PATHS[
    "reference_semantics"
].write_text(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


if not reference_semantics_valid:

    raise AssertionError(
        "D13 Stage 1 reference "
        "semantics are not valid."
    )


print(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 5. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

schema_diagnostics = {
    "valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity
}


if not structurally_evaluable:

    raise ValueError(
        "D13 Branch B output is not "
        "structurally evaluable. "
        "Content validation cannot proceed."
    )


print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 6. Comparison-only canonicalisation
# ============================================================
# FROZEN FROM D13 VALIDATION A.

def parse_numeric(value):

    if (
        value is None
        or isinstance(
            value,
            bool
        )
    ):
        return None

    if isinstance(
        value,
        (int, float)
    ):
        return float(value)

    text = normalise_text(
        value
    )

    if text is None:
        return None

    text = text.replace(
        ",",
        ""
    )

    if re.fullmatch(
        r"[-+]?\d+(?:\.\d+)?",
        text
    ):
        return float(text)

    return None


def compare_field(
    field,
    reference_value,
    extracted_value
):

    if field == "Value":

        reference_numeric = (
            parse_numeric(
                reference_value
            )
        )

        extracted_numeric = (
            parse_numeric(
                extracted_value
            )
        )

        return (
            reference_numeric
            is not None
            and extracted_numeric
            is not None
            and reference_numeric
            == extracted_numeric
        )

    if (
        field
        == "Reporting Period"
    ):

        return (
            canonical_period(
                reference_value
            )
            ==
            canonical_period(
                extracted_value
            )
        )

    if (
        field
        == "Source Location"
    ):

        return (
            canonical_source_location(
                reference_value
            )
            ==
            canonical_source_location(
                extracted_value
            )
        )

    return (
        normalise_text(
            reference_value
        )
        ==
        normalise_text(
            extracted_value
        )
    )

In [ ]:
# ============================================================
# 7. Deterministic one-to-one alignment by Source Location
# ============================================================
# FROZEN FROM D13 VALIDATION A.
#
# Physical workbook Source Location is the sole alignment
# identity. No content field is used to establish correspondence.

def build_identity_index(
    records,
    dataset_name
):

    index = {}
    duplicate_groups = []

    for record_index, record in enumerate(
        records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue

        identity = (
            canonical_source_location(
                record.get(
                    "Source Location"
                )
            )
        )

        if identity in index:

            duplicate_groups.append({
                "dataset":
                    dataset_name,

                "identity":
                    identity,

                "first_record_index":
                    index[
                        identity
                    ][
                        "record_index"
                    ],

                "duplicate_record_index":
                    record_index
            })

        else:

            index[
                identity
            ] = {
                "record_index":
                    record_index,

                "record":
                    record
            }

    return (
        index,
        duplicate_groups
    )


reference_records = (
    reference_df.to_dict(
        "records"
    )
)


reference_index, reference_duplicates = (
    build_identity_index(
        reference_records,
        "Reference"
    )
)


extraction_index, extraction_duplicates = (
    build_identity_index(
        extracted_records,
        "Extraction"
    )
)


alignment_issues = (
    reference_duplicates
    + extraction_duplicates
)


PATHS[
    "alignment_issues"
].write_text(
    json.dumps(
        alignment_issues,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


if reference_duplicates:

    raise AssertionError(
        "Reference Source Location "
        "identity is not unique."
    )


if extraction_duplicates:

    raise AssertionError(
        "Extracted Source Location "
        "identity is not unique."
    )


print(
    "Reference duplicate identities:",
    len(reference_duplicates)
)

print(
    "Extraction duplicate identities:",
    len(extraction_duplicates)
)

In [ ]:
# ============================================================
# 8. Match records and compare fields
# ============================================================

detailed_rows = []
discrepant_rows = []
missing_rows = []
unsupported_rows = []


all_identities = sorted(
    set(
        reference_index.keys()
    )
    |
    set(
        extraction_index.keys()
    ),
    key=lambda x:
        (
            x is None,
            str(x)
        )
)


for identity in all_identities:

    ref_entry = (
        reference_index.get(
            identity
        )
    )

    ext_entry = (
        extraction_index.get(
            identity
        )
    )


    if (
        ref_entry is not None
        and ext_entry is None
    ):

        row = (
            ref_entry[
                "record"
            ].copy()
        )

        row[
            "Reference Record Index"
        ] = (
            ref_entry[
                "record_index"
            ]
        )

        missing_rows.append(
            row
        )

        continue


    if (
        ext_entry is not None
        and ref_entry is None
    ):

        row = (
            ext_entry[
                "record"
            ].copy()
        )

        row[
            "Extracted Record Index"
        ] = (
            ext_entry[
                "record_index"
            ]
        )

        unsupported_rows.append(
            row
        )

        continue


    ref_record = (
        ref_entry[
            "record"
        ]
    )

    ext_record = (
        ext_entry[
            "record"
        ]
    )


    field_results = {
        field:
            compare_field(
                field,
                ref_record.get(
                    field
                ),
                ext_record.get(
                    field
                )
            )

        for field
        in EXPECTED_FIELDS
    }


    primary_correct = all(
        field_results[
            field
        ]

        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )


    identity_correct = all(
        field_results[
            field
        ]

        for field
        in ALIGNMENT_IDENTITY_FIELDS
    )


    # Formal correctness is determined only by
    # PRIMARY_CORRECTNESS_FIELDS.
    fully_correct = (
        primary_correct
    )


    detail = {
        "Reference Record Index":
            ref_entry[
                "record_index"
            ],

        "Extracted Record Index":
            ext_entry[
                "record_index"
            ],

        "Identity Source Location":
            identity,

        "Fully Correct":
            bool(
                fully_correct
            ),

        "Primary Correct":
            bool(
                primary_correct
            ),

        "Alignment Identity Correct":
            bool(
                identity_correct
            )
    }


    for field in EXPECTED_FIELDS:

        detail[
            f"Reference {field}"
        ] = ref_record.get(
            field
        )

        detail[
            f"Extracted {field}"
        ] = ext_record.get(
            field
        )

        detail[
            f"{field} Correct"
        ] = field_results[
            field
        ]


    detailed_rows.append(
        detail
    )


    if not fully_correct:

        discrepant_rows.append(
            detail.copy()
        )


detailed_df = pd.DataFrame(
    detailed_rows
)

discrepant_df = pd.DataFrame(
    discrepant_rows
)

missing_df = pd.DataFrame(
    missing_rows
)

unsupported_df = pd.DataFrame(
    unsupported_rows
)


print(
    "Aligned:",
    len(detailed_df)
)

print(
    "Discrepant:",
    len(discrepant_df)
)

print(
    "Missing:",
    len(missing_df)
)

print(
    "Unsupported/unmatched:",
    len(unsupported_df)
)

In [ ]:
# ============================================================
# 9. Calculate common validation metrics
# ============================================================

reference_count = int(
    len(reference_df)
)

extracted_count = int(
    len(extracted_records)
)

aligned_count = int(
    len(detailed_df)
)

fully_correct_count = (
    int(
        detailed_df[
            "Fully Correct"
        ].sum()
    )
    if aligned_count
    else 0
)

discrepant_count = int(
    len(discrepant_df)
)

missing_count = int(
    len(missing_df)
)

unsupported_count = int(
    len(unsupported_df)
)


completeness = (
    aligned_count
    / reference_count
    if reference_count
    else 0.0
)

missing_rate = (
    missing_count
    / reference_count
    if reference_count
    else 0.0
)


record_precision_exact = (
    fully_correct_count
    / extracted_count
    if extracted_count
    else 0.0
)

record_recall_exact = (
    fully_correct_count
    / reference_count
    if reference_count
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


unsupported_rate = (
    unsupported_count
    / extracted_count
    if extracted_count
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_count
    / aligned_count
    if aligned_count
    else 0.0
)


field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:

    if aligned_count:

        field_accuracy_among_aligned[
            field
        ] = float(
            detailed_df[
                f"{field} Correct"
            ].mean()
        )

    else:

        field_accuracy_among_aligned[
            field
        ] = 0.0


correct_field_instances = int(
    sum(
        detailed_df[
            f"{field} Correct"
        ].sum()

        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )
)


expected_field_instances = int(
    reference_count
    * len(
        PRIMARY_CORRECTNESS_FIELDS
    )
)


field_accuracy = (
    correct_field_instances
    / expected_field_instances
    if expected_field_instances
    else 0.0
)


print(
    "Fully correct records:",
    fully_correct_count
)

print(
    "Discrepant records:",
    discrepant_count
)

print(
    "Completeness:",
    completeness
)

print(
    "Exact record F1:",
    record_f1_exact
)

print(
    "Field accuracy:",
    field_accuracy
)

In [ ]:
# ============================================================
# 10. Category- and reporting-period-level diagnostics
# ============================================================

category_metrics = {}

for category in sorted(
    set(
        reference_df[
            "Category"
        ].astype(str)
    )
):

    ref_category_count = int(
        (
            reference_df[
                "Category"
            ]
            == category
        ).sum()
    )

    ext_category_count = int(
        sum(
            1

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == category
        )
    )


    aligned_category = (
        detailed_df[
            detailed_df[
                "Reference Category"
            ]
            == category
        ]
        if aligned_count
        else pd.DataFrame()
    )


    aligned_category_count = int(
        len(
            aligned_category
        )
    )


    fully_correct_category = (
        int(
            aligned_category[
                "Fully Correct"
            ].sum()
        )
        if aligned_category_count
        else 0
    )


    p = (
        fully_correct_category
        / ext_category_count
        if ext_category_count
        else 0.0
    )

    r = (
        fully_correct_category
        / ref_category_count
        if ref_category_count
        else 0.0
    )

    f1 = (
        2 * p * r
        / (p + r)
        if p + r
        else 0.0
    )


    category_metrics[
        category
    ] = {
        "expected_records":
            ref_category_count,

        "extracted_records":
            ext_category_count,

        "aligned_records":
            aligned_category_count,

        "fully_correct_records":
            fully_correct_category,

        "discrepant_records":
            (
                aligned_category_count
                - fully_correct_category
            ),

        "completeness":
            (
                aligned_category_count
                / ref_category_count
                if ref_category_count
                else 0.0
            ),

        "record_precision_exact":
            p,

        "record_recall_exact":
            r,

        "record_f1_exact":
            f1
    }


year_metrics = {}

for year in SELECTED_YEARS:

    ref_year_count = int(
        (
            reference_df[
                "Reporting Period"
            ].apply(
                canonical_period
            )
            == year
        ).sum()
    )


    ext_year_count = int(
        sum(
            1

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
            and canonical_period(
                record.get(
                    "Reporting Period"
                )
            )
            == year
        )
    )


    aligned_year = (
        detailed_df[
            detailed_df[
                "Reference Reporting Period"
            ].apply(
                canonical_period
            )
            == year
        ]
        if aligned_count
        else pd.DataFrame()
    )


    aligned_year_count = int(
        len(
            aligned_year
        )
    )


    fully_correct_year = (
        int(
            aligned_year[
                "Fully Correct"
            ].sum()
        )
        if aligned_year_count
        else 0
    )


    year_metrics[
        str(year)
    ] = {
        "expected_records":
            ref_year_count,

        "extracted_records":
            ext_year_count,

        "aligned_records":
            aligned_year_count,

        "fully_correct_records":
            fully_correct_year,

        "discrepant_records":
            (
                aligned_year_count
                - fully_correct_year
            )
    }

In [ ]:
# ============================================================
# 11. Build field-level diagnostics
# ============================================================

field_validation_rows = []


for field in EXPECTED_FIELDS:

    correct = (
        int(
            detailed_df[
                f"{field} Correct"
            ].sum()
        )
        if aligned_count
        else 0
    )


    field_validation_rows.append({
        "Field":
            field,

        "used_in_alignment_identity":
            field
            in ALIGNMENT_IDENTITY_FIELDS,

        "used_in_primary_correctness":
            field
            in PRIMARY_CORRECTNESS_FIELDS,

        "Correct":
            correct,

        "Aligned Records":
            aligned_count,

        "Accuracy Among Aligned":
            (
                correct
                / aligned_count
                if aligned_count
                else 0.0
            ),

        "Overall Expected Instances":
            reference_count,

        "Overall Accuracy Against Reference":
            (
                correct
                / reference_count
                if reference_count
                else 0.0
            )
    })


field_validation_df = pd.DataFrame(
    field_validation_rows
)


category_metrics_df = pd.DataFrame([
    {
        "Category":
            category,
        **metrics
    }

    for category, metrics
    in category_metrics.items()
])


year_metrics_df = pd.DataFrame([
    {
        "Reporting Period":
            year,
        **metrics
    }

    for year, metrics
    in year_metrics.items()
])


fully_correct_df = (
    detailed_df[
        detailed_df[
            "Fully Correct"
        ]
    ].copy()
)


display(
    field_validation_df
)

In [ ]:
# ============================================================
# 12. Build final Branch B validation summary
# ============================================================

content_diagnostics = {
    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_category_counts_valid":
        bool(
            reference_category_counts_valid
        ),

    "reference_year_counts_valid":
        bool(
            reference_year_counts_valid
        ),

    "reference_flag_count_valid":
        bool(
            reference_flag_count_valid
        ),

    "reference_flag_values_valid":
        bool(
            reference_flag_values_valid
        ),

    "extraction_record_count_valid":
        bool(
            extracted_count
            == EXPECTED_RECORD_COUNT
        ),

    "extraction_category_counts_valid":
        (
            dict(
                Counter(
                    record.get(
                        "Category"
                    )

                    for record
                    in extracted_records

                    if isinstance(
                        record,
                        dict
                    )
                )
            )
            == EXPECTED_CATEGORY_COUNTS
        ),

    "branch_B_scope_complete":
        technical_diagnostics.get(
            "scope_complete"
        ),

    "branch_B_content_diagnostics":
        technical_diagnostics.get(
            "content_diagnostics"
        ),

    "reference_identity_unique":
        bool(
            reference_identity_unique
        ),

    "extraction_duplicate_identity_count":
        int(
            len(
                extraction_duplicates
            )
        ),

    "alignment_issue_count":
        int(
            len(
                alignment_issues
            )
        )
}


comparison_rules = {
    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "value":
        (
            "Exact represented numeric equality after "
            "deterministic parsing; no tolerance, rounding, "
            "interpolation, rescaling or conversion."
        ),

    "category_topic_unit":
        (
            "Conservative normalised exact agreement "
            "with the fixed Stage 1 values."
        ),

    "description":
        (
            "Normalised exact agreement using the "
            "fixed D13 description template."
        ),

    "reporting_period":
        "Canonical exact year agreement.",

    "source_location":
        (
            "Physical workbook value-cell coordinate "
            "defines observation identity and is used "
            "for alignment only."
        ),

    "equivalence_rules_frozen_from_branch_A":
        True
}


summary = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "reference_records":
        reference_count,

    "extracted_records":
        extracted_count,

    "aligned_records":
        aligned_count,

    "fully_correct_records":
        fully_correct_count,

    "discrepant_records":
        discrepant_count,

    "missing_records":
        missing_count,

    "unsupported_extracted_records":
        unsupported_count,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision_exact,
            4
        ),

    "record_recall_exact":
        round(
            record_recall_exact,
            4
        ),

    "record_f1_exact":
        round(
            record_f1_exact,
            4
        ),

    "unsupported_rate":
        round(
            unsupported_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate_among_aligned,
            4
        ),

    "field_accuracy":
        round(
            field_accuracy,
            4
        ),

    "field_accuracy_among_aligned": {
        field:
            round(
                value,
                4
            )

        for field, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "structurally_evaluable":
        structurally_evaluable,

    "content_diagnostics":
        content_diagnostics,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "comparison_rules_frozen_from_branch_A":
        True,

    "comparison_rules":
        comparison_rules,

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(
                reference_semantics_valid
            ),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False
    },

    "category_metrics":
        category_metrics,

    "year_metrics":
        year_metrics,

    "input_provenance":
        input_provenance
}


print(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 13. Validation integrity checks
# ============================================================

assert (
    aligned_count
    + missing_count
    == reference_count
)

assert (
    aligned_count
    + unsupported_count
    == extracted_count
)

assert (
    fully_correct_count
    + discrepant_count
    == aligned_count
)


for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "unsupported_rate":
        unsupported_rate,

    "discrepancy_rate":
        discrepancy_rate_among_aligned,

    "field_accuracy":
        field_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )


assert (
    reference_semantics_valid
)


print(
    "Validation integrity checks passed."
)

In [ ]:
# ============================================================
# 14. Export Validation B artefacts
# ============================================================

detailed_df.to_csv(
    PATHS[
        "detailed"
    ],
    index=False,
    encoding="utf-8-sig"
)


fully_correct_df.to_csv(
    PATHS[
        "fully_correct"
    ],
    index=False,
    encoding="utf-8-sig"
)


discrepant_df.to_csv(
    PATHS[
        "discrepant"
    ],
    index=False,
    encoding="utf-8-sig"
)


missing_df.to_csv(
    PATHS[
        "missing"
    ],
    index=False,
    encoding="utf-8-sig"
)


unsupported_df.to_csv(
    PATHS[
        "unsupported"
    ],
    index=False,
    encoding="utf-8-sig"
)


field_validation_df.to_csv(
    PATHS[
        "field_validation"
    ],
    index=False,
    encoding="utf-8-sig"
)


category_metrics_df.to_csv(
    PATHS[
        "category_metrics"
    ],
    index=False,
    encoding="utf-8-sig"
)


year_metrics_df.to_csv(
    PATHS[
        "year_metrics"
    ],
    index=False,
    encoding="utf-8-sig"
)


PATHS[
    "summary"
].write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "D13 Validation B artefacts saved."
)

In [ ]:
# ============================================================
# 15. Download generated validation artefacts
# ============================================================

for output_path in (
    PATHS.values()
):

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )


for output_path in (
    PATHS.values()
):

    if output_path.exists():

        files.download(
            output_path
        )